<a href="https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1) Two paper findings + my methodology questions

### Finding 1 — Health-score feature importance

The FlyRank research paper reports that a Random Forest model predicting the Health Score assigned the highest feature importance to Average Position (43%), followed by Impressions (32%) and Scroll Depth (15%). The paper explicitly notes that the Health Score is partly constructed from inputs such as position and impressions, so these importance values are descriptive rather than causal.

**Methodology question:**  
Because the Health Score itself includes impressions, position, CTR, and scroll depth, how much of the reported feature importance reflects the mathematical construction of the target rather than an independently validated predictive relationship?

**Why this matters:**  
A feature can appear highly important when it is also part of how the target is constructed. I would therefore interpret this result as observed model behavior within the study rather than evidence that changing one feature would independently cause a higher Health Score.

---

### Finding 2 — Logistic regression holdout accuracy

The paper reports 71% holdout accuracy for a logistic regression model describing which sampled features separate growing from declining pages. The methodology section describes an 80/20 split for the ML pipeline and states that the ML analysis is exploratory and secondary to the direct portfolio evidence.

**Methodology question:**  
Does the 80/20 holdout design adequately test generalization across clients and time, given that the study uses observational content data and that client-level or temporal differences may be represented in both the training and holdout populations?

**Why this matters:**  
A holdout score can be useful evidence of measured performance, but the validation design determines what kind of generalization the score supports. A grouped-by-client or time-aware evaluation could provide a different estimate if content from the same clients or similar time periods appears on both sides of the split.

---

### Review posture

These questions are intended as constructive methodology checks rather than judgments of the research. The goal is to understand what the reported evidence supports and where additional validation could strengthen the interpretation.

## 2. My model under an honest split (before/after)

### 2.1 Reproducing the Week-5 evaluation

I first reproduce the Week-5 evaluation setup so that this audit has a directly comparable baseline.

The Week-5 model already used a client-grouped evaluation, so this audit does not present a random-to-grouped improvement that did not occur. The starter snapshot also does not contain a suitable row-level reporting date for a genuine time-aware train/test evaluation. Instead, I re-ran the model using an independent client-grouped validation split with zero client overlap. I treat the difference between the Week-5 grouped result and this independent grouped result as a robustness observation, not as evidence that changing the split method caused the metric difference. This avoids fabricating a baseline and keeps the evaluation aligned with the available data.

In [180]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit

DATA_URL = "https://raw.githubusercontent.com/nomanamir20/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)

# Keep the same modeling lane as Week 5
df = df[df["content_type"] == "keyword article"].copy()

print("Keyword article lane shape:", df.shape)

# Recreate the Week-5 target
df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nOverall decline rate:")
print(f"{df['is_declining_label'].mean():.4f}")

Dataset shape: (30000, 44)
Keyword article lane shape: (27207, 44)

Target distribution:
is_declining_label
1    15262
0    11945
Name: count, dtype: int64

Overall decline rate:
0.5610


In [181]:
# Reproduce the exact Week-5 split:
# 80/20 grouped by client_id, random_state=42

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Training rows:", len(train))
print("Test rows:", len(test))

print("\nTraining clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

overlap = set(train["client_id"]).intersection(
    set(test["client_id"])
)

print("Client overlap:", len(overlap))

print("\nTraining decline rate:")
print(f"{train['is_declining_label'].mean():.4f}")

print("\nTest decline rate:")
print(f"{test['is_declining_label'].mean():.4f}")

Training rows: 21425
Test rows: 5782

Training clients: 24
Test clients: 7
Client overlap: 0

Training decline rate:
0.5762

Test decline rate:
0.5043


### 2.2 Week-5 grouped evaluation baseline

The Week-5 notebook already used a grouped train/test split by `client_id`. Therefore, the purpose of this section is not to claim that Week 5 used a random row-level split.

Instead, I reproduce the exact Week-5 grouped evaluation so that the later audit has a measured **before** result.

The model uses Logistic Regression with the same feature set and preprocessing approach used in Week 5. The target is `is_declining_label`.

Because the target is derived from `trend_direction`, I will explicitly exclude `trend_direction` and `trend_pct` from model features.

In [182]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score


numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]

feature_columns = numeric_features + categorical_features

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total model features:", len(feature_columns))

# Explicit leakage check
for forbidden in [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id"
]:
    print(
        f"{forbidden}:",
        "IN FEATURES" if forbidden in feature_columns else "excluded"
    )

Numeric features: 22
Categorical features: 8
Total model features: 30
trend_direction: excluded
trend_pct: excluded
is_declining_label: excluded
client_id: excluded
content_id: excluded


In [183]:
X_train = train[feature_columns].copy()
X_test = test[feature_columns].copy()

y_train = train["is_declining_label"]
y_test = test["is_declining_label"]


numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)


model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)


model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]

print("Week-5 Logistic Regression reproduced successfully.")
print("Number of test predictions:", len(model_scores))

Week-5 Logistic Regression reproduced successfully.
Number of test predictions: 5782


In [184]:
def precision_at_k(y_true, scores, k):
    ranking = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    ranking = ranking.sort_values(
        "score",
        ascending=False
    )

    top_k = ranking.head(min(k, len(ranking)))

    return float(top_k["y_true"].mean())


k_values = [20, 50, 100]

before_results = []

for k in k_values:
    before_results.append({
        "Metric": f"Precision@{k}",
        "Before": precision_at_k(
            y_test,
            model_scores,
            k
        )
    })


before_auc = roc_auc_score(
    y_test,
    model_scores
)

before_ap = average_precision_score(
    y_test,
    model_scores
)

before_results.append({
    "Metric": "ROC-AUC",
    "Before": before_auc
})

before_results.append({
    "Metric": "Average Precision",
    "Before": before_ap
})


before_table = pd.DataFrame(before_results)

print("WEEK-5 BEFORE RESULTS")
display(before_table.round(4))

WEEK-5 BEFORE RESULTS


,Metric,Before
0,Precision@20,0.8500
1,Precision@50,0.7400
2,Precision@100,0.7200
3,ROC-AUC,0.6161
4,Average Precision,0.6097


In [185]:
print("Available model variables:")

for name in ["model", "best_model", "clf", "classifier", "pipeline"]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj)}")

Available model variables:
model: <class 'sklearn.pipeline.Pipeline'>


In [186]:
print("Data variables:")

for name in ["df", "data", "dataset", "X", "y", "X_train", "y_train"]:
    if name in globals():
        obj = globals()[name]
        try:
            print(f"{name}: {type(obj)} | shape={obj.shape}")
        except Exception:
            print(f"{name}: {type(obj)}")

Data variables:
df: <class 'pandas.core.frame.DataFrame'> | shape=(27207, 45)
X_train: <class 'pandas.core.frame.DataFrame'> | shape=(21425, 30)
y_train: <class 'pandas.core.series.Series'> | shape=(21425,)


In [187]:
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

# Recover client groups using the original row indices
groups = df.loc[X_train.index, "client_id"]

# Honest grouped split:
# no client can appear in both train and validation
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, valid_idx = next(
    splitter.split(X_train, y_train, groups=groups)
)

X_group_train = X_train.iloc[train_idx]
X_group_valid = X_train.iloc[valid_idx]

y_group_train = y_train.iloc[train_idx]
y_group_valid = y_train.iloc[valid_idx]

groups_train = groups.iloc[train_idx]
groups_valid = groups.iloc[valid_idx]

print("Grouped train shape:", X_group_train.shape)
print("Grouped validation shape:", X_group_valid.shape)
print("Training clients:", groups_train.nunique())
print("Validation clients:", groups_valid.nunique())

# Verify that no client appears in both sets
overlap = set(groups_train.unique()) & set(groups_valid.unique())
print("Client overlap:", len(overlap))

Grouped train shape: (18636, 30)
Grouped validation shape: (2789, 30)
Training clients: 19
Validation clients: 5
Client overlap: 0


In [188]:
grouped_model = clone(model)

grouped_model.fit(
    X_group_train,
    y_group_train
)

grouped_scores = grouped_model.predict_proba(X_group_valid)[:, 1]

grouped_metrics = {
    "Precision@20": precision_at_k(
        y_group_valid,
        grouped_scores,
        20
    ),
    "Precision@50": precision_at_k(
        y_group_valid,
        grouped_scores,
        50
    ),
    "Precision@100": precision_at_k(
        y_group_valid,
        grouped_scores,
        100
    ),
    "ROC-AUC": roc_auc_score(
        y_group_valid,
        grouped_scores
    ),
    "Average Precision": average_precision_score(
        y_group_valid,
        grouped_scores
    ),
}

pd.DataFrame(
    [grouped_metrics],
    index=["Grouped client holdout"]
)

,Precision@20,Precision@50,Precision@100,ROC-AUC,Average Precision
Grouped client holdout,0.95,0.96,0.96,0.669583,0.856826


In [189]:
base_rate = float(y_group_valid.mean())

print(f"Validation positive rate: {base_rate:.4f}")
print(f"Validation positive rate: {base_rate * 100:.2f}%")

Validation positive rate: 0.7418
Validation positive rate: 74.18%


### Validation base rate

**Validation positive rate: 74.18%** of the client-grouped validation examples have `is_declining_label = 1`.

This base rate provides the naive positive-class context for interpreting the ranking and classification metrics below. The reported metrics should therefore be read as measured performance on this specific validation population, rather than as standalone evidence of general deployment performance.

What this demonstrates:
The important validation improvement demonstrated here is the separation of clients between training and validation, together with an independent grouped re-run. The results are therefore interpreted as measured evidence about performance on unseen clients, rather than as a claim of future deployment performance.

In [190]:
before_metrics = {
    "Precision@20": 0.8500,
    "Precision@50": 0.7400,
    "Precision@100": 0.7200,
    "ROC-AUC": 0.6161,
    "Average Precision": 0.6097,
}

comparison = pd.DataFrame({
    "Before (Week-5)": before_metrics,
    "After (Client-grouped)": grouped_metrics,
})

comparison["Change"] = (
    comparison["After (Client-grouped)"]
    - comparison["Before (Week-5)"]
)

comparison

,Before (Week-5),After (Client-grouped),Change
Precision@20,0.8500,0.950000,0.100000
Precision@50,0.7400,0.960000,0.220000
Precision@100,0.7200,0.960000,0.240000
ROC-AUC,0.6161,0.669583,0.053483
Average Precision,0.6097,0.856826,0.247126


### Section 2 conclusion

The Week-5 model was first evaluated using its original client-grouped holdout. I then evaluated the same model approach using a separately generated client-grouped validation split within the Week-5 training population.

The secondary client-grouped validation produced the following measured results:

* Precision@20: 0.9500
* Precision@50: 0.9600
* Precision@100: 0.9600
* ROC-AUC: 0.6696
* Average Precision: 0.8568

The Week-5 recorded results were:

* Precision@20: 0.8500
* Precision@50: 0.7400
* Precision@100: 0.7200
* ROC-AUC: 0.6161
* Average Precision: 0.6097

The secondary validation therefore produced higher measured metrics in this particular split. However, because both evaluations use client-grouped validation, this comparison should not be interpreted as evidence that grouping alone caused the improvement.

The results are best treated as an observed comparison between two client-level evaluation samples. They provide directional evidence about the model's behavior under a separate held-out client population, but they do not establish generalization to all unseen clients or future time periods.

Additional time-aware validation and a properly sealed future holdout would provide stronger evidence for deployment-oriented claims.

#Validation conclusion:
The independent client-grouped evaluation confirms that the model can be measured on clients that are excluded from training. The validation positive rate was 74.18%, which is reported alongside the model metrics as the base-rate context. Because the Week-5 evaluation was also client-grouped and the starter snapshot lacks an appropriate time variable, I do not claim that the validation method itself caused an improvement. The observed metric differences are treated as a robustness check and directional evidence only.

###Why no split-method improvement is claimed:
The Week-5 model already used client-grouped validation, so this audit does not fabricate a random-to-grouped comparison. The starter snapshot also lacks a suitable row-level reporting date for a genuine time-aware evaluation. Therefore, the before/after table compares two independently generated client-grouped evaluations and is interpreted only as a robustness observation. The observed metric differences are not attributed to the grouping method itself.

## 3. Leakage Audit

Before trusting the client-grouped results, I audited the feature set for information that could directly or indirectly reveal the target.

The target `is_declining_label` is derived from `trend_direction`, which itself is derived from `trend_pct`. Therefore, `trend_direction` and `trend_pct` must never be used as model features.

I also checked for identifier leakage (`content_id`, `client_id`), decision-derived fields, and features that overlap with the label-generation window.

The purpose of this audit is not to prove that the model is perfect, but to identify whether the measured performance could be inflated by information that would not be legitimately available at prediction time.

In [191]:
# Verify the label source

label_column = "is_declining_label"

print("Label column:", label_column)

if "trend_direction" in df.columns:
    expected_label = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

    print("Label matches trend_direction:",
          bool((df[label_column].astype(int) == expected_label).all()))

if "trend_pct" in df.columns:
    print("trend_pct exists in dataframe:", True)

print("\nForbidden label-source columns:")
print(["trend_direction", "trend_pct"])

Label column: is_declining_label
Label matches trend_direction: True
trend_pct exists in dataframe: True

Forbidden label-source columns:
['trend_direction', 'trend_pct']


In [192]:
# Audit the model feature lists

forbidden_features = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
}

configured_features = (
    set(numeric_features)
    | set(categorical_features)
)

violations = sorted(configured_features & forbidden_features)

print("Number of configured model features:", len(configured_features))
print("Forbidden features found in model configuration:", len(violations))

if violations:
    print("LEAKAGE WARNING:", violations)
else:
    print("PASS: No forbidden label-source or identifier columns are configured as model features.")

Number of configured model features: 30
Forbidden features found in model configuration: 0
PASS: No forbidden label-source or identifier columns are configured as model features.


### Feature provenance audit

The main leakage risks identified in the data dictionary are:

- `trend_direction` — directly used to construct the target.
- `trend_pct` — directly used to construct `trend_direction`.
- `content_id` and `client_id` — identifiers that could allow memorization of entities rather than learning generalizable patterns.
- Product/system decision fields — should not be used as predictive inputs when they encode an existing decision.
- Recent activity windows — must be checked against the prediction/label window because overlapping time periods can expose future information.

The current model configuration excludes the explicit label-source columns and identifiers.

However, the starter dataset is a retrospective 90-day snapshot. Therefore, the remaining activity features should be interpreted as observed associations and decision-support signals rather than proof of future predictive performance.

In [193]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Deliberate leakage sanity check
# This is intentionally NOT a valid model feature.
# It directly copies the target to verify that the evaluation harness
# exposes an obvious label-leakage scenario.

leak_check = pd.DataFrame({
    "leaky_target_copy": df["is_declining_label"].astype(int)
})

leak_model = LogisticRegression(max_iter=1000)
leak_model.fit(leak_check, df["is_declining_label"])

leak_scores = leak_model.predict_proba(leak_check)[:, 1]

leak_roc_auc = roc_auc_score(df["is_declining_label"], leak_scores)
leak_average_precision = average_precision_score(df["is_declining_label"], leak_scores)

print("Deliberate leakage sanity check")
print(f"ROC-AUC: {leak_roc_auc:.4f}")
print(f"Average Precision: {leak_average_precision:.4f}")

Deliberate leakage sanity check
ROC-AUC: 1.0000
Average Precision: 1.0000


### Deliberate leakage sanity check

I intentionally added a feature that directly copies the target label. This feature is not a valid model input and is used only to test whether the evaluation harness exposes an obvious leakage scenario.

The resulting near-perfect metrics are expected because the model has direct access to the answer.

This confirms that the evaluation setup can detect an artificial label-leakage case. The leaky feature is excluded from all legitimate model evaluation and is not considered a usable feature.

### Temporal leakage check

The label is based on the change in impressions between the most recent 30-day period and the preceding 30-day period.

The starter dataset is a retrospective 90-day snapshot rather than a true deployment-time prediction dataset. Some activity features therefore describe the same observation window from which the target was constructed.

This means the client-grouped split improves protection against client memorization, but it does not by itself establish a true future-prediction evaluation.

For that reason, the results are treated as measured performance on the available snapshot and as decision-support evidence, rather than evidence that the model will achieve the same performance on future unseen periods.

In [194]:
# Final leakage audit summary

leakage_audit = pd.DataFrame([
    {
        "Risk": "Label-derived features",
        "Fields": "trend_direction, trend_pct",
        "Status": "Excluded",
        "Assessment": "Direct label-source leakage prevented"
    },
    {
        "Risk": "Identifier leakage",
        "Fields": "content_id, client_id",
        "Status": "Excluded",
        "Assessment": "Identifiers used for grouping only"
    },
    {
        "Risk": "Product/system decision fields",
        "Fields": "Existing decision flags/scores",
        "Status": "Excluded",
        "Assessment": "Not used as predictive inputs"
    },
    {
        "Risk": "Temporal overlap",
        "Fields": "90-day activity features",
        "Status": "Limitation",
        "Assessment": "Snapshot does not establish true future prediction"
    },
    {
        "Risk": "Deliberate leakage test",
        "Fields": "Target copied as feature",
        "Status": "Test only",
        "Assessment": "Demonstrates evaluation can expose obvious leakage"
    },
])

display(leakage_audit)

,Risk,Fields,Status,Assessment
0,Label-derived features,"trend_direction, trend_pct",Excluded,Direct label-source leakage prevented
1,Identifier leakage,"content_id, client_id",Excluded,Identifiers used for grouping only
2,Product/system decision fields,Existing decision flags/scores,Excluded,Not used as predictive inputs
3,Temporal overlap,90-day activity features,Limitation,Snapshot does not establish true future predic...
4,Deliberate leakage test,Target copied as feature,Test only,Demonstrates evaluation can expose obvious lea...


In [195]:
# Real failure examples from the client-grouped validation set

failure_examples = X_group_valid.copy()

failure_examples["actual"] = np.asarray(y_group_valid)
failure_examples["predicted_probability"] = np.asarray(grouped_scores)
failure_examples["predicted"] = (
    failure_examples["predicted_probability"] >= 0.5
).astype(int)

failure_examples["client_id"] = groups_valid.to_numpy()

# False positives:
# Model predicted decline, but the actual label was not decline.
false_positives = failure_examples[
    (failure_examples["predicted"] == 1) &
    (failure_examples["actual"] == 0)
].copy()

# False negatives:
# Model predicted no decline, but the actual label was decline.
false_negatives = failure_examples[
    (failure_examples["predicted"] == 0) &
    (failure_examples["actual"] == 1)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

# Show representative false positives
print("\nRepresentative false positives:")
display(
    false_positives[
        [
            "client_id",
            "actual",
            "predicted",
            "predicted_probability",
            "content_type"
        ]
    ]
    .sort_values("predicted_probability", ascending=False)
    .head(5)
)

# Show representative false negatives
print("\nRepresentative false negatives:")
display(
    false_negatives[
        [
            "client_id",
            "actual",
            "predicted",
            "predicted_probability",
            "content_type"
        ]
    ]
    .sort_values("predicted_probability", ascending=True)
    .head(5)
)

False positives: 338
False negatives: 606

Representative false positives:


,client_id,actual,predicted,predicted_probability,content_type
14610,client_02d20bbd7e,0,1,0.767576,keyword article
10706,client_02d20bbd7e,0,1,0.734353,keyword article
26272,client_7f2253d7e2,0,1,0.730021,keyword article
29640,client_02d20bbd7e,0,1,0.725862,keyword article
17363,client_7f2253d7e2,0,1,0.710394,keyword article



Representative false negatives:


,client_id,actual,predicted,predicted_probability,content_type
2797,client_7f2253d7e2,1,0,0.016299,keyword article
1448,client_7f2253d7e2,1,0,0.036759,keyword article
12508,client_7f2253d7e2,1,0,0.037023,keyword article
23671,client_7f2253d7e2,1,0,0.038680,keyword article
25590,client_7f2253d7e2,1,0,0.045423,keyword article


### Observed failure patterns

The client-grouped validation set produced 338 false positives and 606 false negatives.

The false-positive examples show cases where the model assigned a relatively high probability of decline even though the observed label was 0. The false-negative examples show cases where the model assigned a low probability of decline even though the observed label was 1.

These are observed failure cases from this validation sample. They demonstrate that the model can make both types of errors even under the client-grouped validation design. The examples are used for model diagnosis and decision-support rather than as evidence of a universal failure pattern.

The repeated appearance of some client IDs also reinforces why client-grouped validation is important: observations from the same client can share characteristics that may affect predictions. These examples therefore should be interpreted within the held-out validation population rather than generalized to all clients.

### Failure examples

The validation predictions were inspected for concrete failure cases rather than relying only on aggregate metrics.

**False positives** are cases where the model predicted a decline but the observed label was not a decline. These cases show where the model may flag content that does not meet the label definition.

**False negatives** are cases where the observed label indicates decline but the model did not predict decline. These cases show where the model can miss content that meets the label definition.

These examples are useful for understanding the model's limitations and for identifying patterns that aggregate metrics may hide. The examples should be interpreted as **observed failure cases in this validation sample**, not as evidence that the model will fail in the same way on every future dataset.

The failure analysis also reinforces the importance of evaluating the model using an honest client-grouped split and reviewing individual predictions alongside aggregate metrics.


## 4. Claim Rewrite

### Original claim

The Week-5 model performs well at identifying declining content and can be used to prioritize content for action.

### Evidence check

The Week-5 results provide the baseline metrics for comparison, while this audit re-evaluated the model using a client-grouped validation split. Under the client-grouped validation, the measured metrics were:

* Precision@20: 0.9500
* Precision@50: 0.9600
* Precision@100: 0.9600
* ROC-AUC: 0.6696
* Average Precision: 0.8568

Compared with the Week-5 results, the measured values increased under the client-grouped evaluation. However, this result comes from the specific validation split used in this notebook and does not by itself establish broad production performance or universal generalization.

### Rewritten claim

The client-grouped evaluation **observed higher measured ranking and classification metrics** than the Week-5 evaluation on the validation data used in this audit. In particular, Precision@20, Precision@50, Precision@100, ROC-AUC, and Average Precision were higher under the grouped split.

These results provide **directional evidence** that the model can support prioritization of potentially declining content in a held-out client-grouped validation setting. The model should therefore be treated as **decision-support**, rather than as a definitive predictor of future content performance.

Further validation across additional clients, time periods, and a properly sealed future holdout would be needed before making stronger claims about deployment performance.

### Claim-language rule used in this audit

Throughout this notebook, claims are limited to what was **observed** and **measured** in the available validation data. Terms such as "guarantees," "proves," "always," "production-ready," and "highly accurate" are avoided because the current evaluation does not support those conclusions.

###Revised claim:
The Week-5 model showed measured ability to prioritize declining content in the evaluated client-grouped validation data. The results provide directional evidence that the model may support content-prioritization decisions, but they do not establish production performance, causal impact, or generalization to unseen future periods. The observed results should therefore be treated as decision-support evidence rather than a guarantee of performance.

###Why I changed the claim:
The original wording went beyond what the evaluation directly measured. The audit showed that the model was evaluated on held-out clients, but the starter dataset does not provide a suitable time variable for a genuine future-period evaluation. I therefore changed the language to emphasize what was observed and measured, while avoiding unsupported claims about production readiness, causal effects, or future performance.

## 5. Self-check

This final checklist verifies that the validation and claim audit covers the required evidence and follows the public-safe language standard.

- [x] **Two research-paper findings reviewed:** Two findings were identified and paired with constructive methodology questions about label provenance and validation design.

- [x] **Honest validation performed:** The Week-5 model was re-evaluated using a client-grouped split so that the same client does not appear across the training and evaluation groups.

- [x] **Before/after comparison included:** The original Week-5 client-grouped results were compared with an independently generated client-grouped validation result. Because Week 5 already used client grouping, this audit does not claim a random-to-grouped improvement.

- [x] **Validation base rate reported:** The positive-label rate of the client-grouped validation population was reported alongside the evaluation results to provide context for interpreting the measured metrics.

- [x] **Label leakage checked:** `trend_direction` and `trend_pct` were treated as label-derived information and excluded from legitimate model features.

- [x] **Identifier leakage checked:** `client_id` and `content_id` were treated as identifiers for grouping/joining rather than model features.

- [x] **Deliberate leakage sanity check performed:** A feature that directly copied the target was intentionally introduced as a diagnostic test. The resulting ROC-AUC and Average Precision of 1.0000 confirmed that the evaluation setup can expose an obvious direct-leakage scenario. The artificial feature was not used in the legitimate model.

- [x] **Real failure examples reviewed:** Representative false positives and false negatives were inspected from the client-grouped evaluation.

- [x] **Claims rewritten to match evidence:** Stronger Week-5 claims were replaced with measured, observed, directional, and decision-support language.

- [x] **Deployment claims avoided:** The results are not presented as proof of production readiness, universal accuracy, or guaranteed performance on future clients or future time periods.

- [x] **Limitations acknowledged:** Additional validation across more clients, time periods, and a properly sealed future holdout would be needed before making stronger generalization or deployment claims.

### Final assessment

The audit provides a more rigorous view of the Week-5 model than the original evaluation alone. The independent client-grouped validation, leakage checks, failure examples, and claim rewrite provide evidence about the model under a stricter audit process while making the limitations explicit.

The appropriate interpretation is **directional decision support based on the measured evaluation**, rather than a guarantee that the model will predict future content decline reliably in every deployment setting.

###Final audit conclusion:
The notebook completed the required validation and research-claim audit. The model was evaluated using client-grouped validation with no client overlap, and the independent grouped re-run provides a robustness check. The 74.18% validation positive rate is reported as base-rate context. The leakage audit found no forbidden model features, while the deliberate leakage test confirmed that a target-derived feature produces artificially perfect performance. Real false-positive and false-negative examples were inspected. Claims were rewritten to describe observed, measured, directional, and decision-support evidence without implying production readiness, causal impact, or guaranteed future performance. Because the Week-5 model already used client-grouped validation and the starter snapshot lacks a suitable row-level time variable, no unsupported random-to-grouped or time-aware improvement is claimed.